In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import VBox, HBox, FloatSlider, Output, FloatLogSlider, Button, Checkbox, Layout, IntSlider
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
from scipy.interpolate import RegularGridInterpolator
import json
from scipy.special import jv

%matplotlib inline

In [2]:


def make_array_file_widgets(get_array, set_array, default_path="circle_array8.json"):
    """
    Create widgets for saving/loading a 2D Python array.

    Parameters
    ----------
    get_array : callable
        Function returning the current 2D array.
    set_array : callable
        Function accepting a new 2D array and updating your editor/display.
    default_path : str
        Default filename/path to save to or load from.
    """

    path_box = widgets.Text(
        value=default_path,
        description="File:",
        layout=widgets.Layout(width="400px")
    )

    save_button = widgets.Button(
        description="Save array",
        button_style="success",
        icon="save"
    )

    load_button = widgets.Button(
        description="Load array",
        button_style="info",
        icon="folder-open"
    )

    status = widgets.Output()

    def save_array(_):
        with status:
            status.clear_output()
            try:
                array = get_array()
                with open(path_box.value, "w", encoding="utf-8") as f:
                    json.dump(array, f, indent=2)
                print(f"Saved array to {path_box.value}")
            except Exception as e:
                print(f"Error saving array: {e}")

    def load_array(_):
        with status:
            status.clear_output()
            try:
                with open(path_box.value, "r", encoding="utf-8") as f:
                    array = json.load(f)

                set_array(array)
                print(f"Loaded array from {path_box.value}")
            except Exception as e:
                print(f"Error loading array: {e}")

    save_button.on_click(save_array)
    load_button.on_click(load_array)

    return widgets.VBox([
        widgets.HBox([path_box, save_button, load_button]),
        status
    ]), load_array

In [3]:
c = 344
n = 128
fig_width = 11
fov = np.pi / 2
todeg = 1 / np.pi * 180
MAX_CROPAC = 4
MAX_MICS = 16

polar_pts = np.meshgrid(
    np.linspace(-np.pi, np.pi, 7 * n),
    np.linspace(0, fov * todeg, n))
cart_pts = np.stack((
    np.cos(polar_pts[0]) * polar_pts[1] / todeg,
    np.sin(polar_pts[0]) * polar_pts[1] / todeg), axis = -1)

ffov = fov + 1e-5
x = np.linspace(-ffov, ffov, n)
y = np.linspace(-ffov, ffov, n)

plot_points = (polar_pts, cart_pts, (x, y))

w_f = FloatLogSlider(
    value=4000, base = 10, min=2, max=np.log10(6000),step=0.001,
    description="Frequency", continuous_update=True, readout_format=".1f"
)
w_idx = IntSlider(
    value=0, min=0, max=MAX_MICS, step=1,
    description="Index", continuous_update=True, readout_format=".0f"
)
w_y = FloatSlider(
    value=0, min=0, max=0.32, step=0.001,
    description="Y", continuous_update=True, readout_format=".3f"
)
w_x = FloatSlider(
    value=0, min=0, max=0.277, step=0.001,
    description="X", continuous_update=True, readout_format=".3f"
)
w_cropac_low = Checkbox(
    value=True,
    description="Low frequency",
)
w_cropac_idx = IntSlider(
    value=0, min=0, max=MAX_CROPAC, step=1,
    description="Cropac", continuous_update=True
)
w_checkbox_list1 = [
    Checkbox(
        description=f"{i}",
        indent=False,
        layout=Layout(width='36px', height='24px', margin='0 2px 0 0')
    )
    for i in range(MAX_MICS)
]
w_checkbox_list2 = [
    Checkbox(
        description=f"{i}",
        indent=False,
        layout=Layout(width='36px', height='24px', margin='0 2px 0 0')
    )
    for i in range(MAX_MICS)
]
w_new = Button(icon="plus", button_style="success",layout=widgets.Layout(width="40px"))
w_delete = Button(icon="trash", button_style="danger",layout=widgets.Layout(width="40px"))

class ArrayState:
    positions = [[0, 0]]
    cropac = {}

class State:
    array = ArrayState()
    do_update=True

state = State()

for w in (w_f, w_x, w_y):
     w.layout.width = f"{fig_width * 80 - 50}px"
    
w_idx.layout.width = f"{fig_width * 80 - 160}px"

# Low frequency CroPaC for an octagon microphone array

def planar_directions(directions):
    return np.atan2(directions[:,0], directions[:,1])

def low_cropac_beampattern(look_direction, steering_vectors, mic_array, k):

    m = steering_vectors.shape[-1]
    steering_vectors = np.reshape(steering_vectors, (n*n, m))
    mp = np.sum(mic_array, axis=1) / m
    mic_array = mic_array - mp[:,None]

    mic = planar_directions(mic_array.T)
    dir = planar_directions(look_direction[None,:])
    relative_angles = dir[0] - mic

    r = np.sqrt(np.sum(mic_array[:,0]**2))

    o1 = 1
    o2 = 2

    kr1 = min(k * r, 1.8411837813)
    kr2 = min(k * r, 3.0542369282)
    U = 2 * np.cos(o1 * relative_angles) / ((1j**o1) * jv(o1, kr1) + (1j**(-o1)) * jv(-o1, kr1)) / m
    V = 2 * np.cos(o2 * relative_angles) / ((1j**o2) * jv(o2, kr2) + (1j**(-o2)) * jv(-o2, kr2)) / m

    z1 = steering_vectors @ U
    z2 = steering_vectors @ V

    z1 = np.reshape(z1, (n, n))
    z2 = np.reshape(z2, (n, n))

    return k * r, z1, z2, np.maximum(0, np.real(z1 * np.conj(z2)))

out = Output()
def update_plot(change=None):

    if not state.do_update:
        return

    fc = w_f.value
    pattern = np.array(state.array.positions).T

    # a = np.exp(2j * np.pi / 16 * (np.arange(16) + 0.5)) * 0.1
    # a = np.array([np.real(a) + 0.276 / 2, np.imag(a) + 0.32 / 2])
    # pattern = a

    idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))

    patternX, patternY = pattern
    pattern = pattern.transpose()

    k = 2 * np.pi * fc / c

    xv, yv = np.meshgrid(x, y)
    thetav = np.atan2(yv, xv)
    rv = np.sqrt(xv*xv + yv*yv)
    
    p = np.stack((k * np.cos(thetav) * np.sin(rv), k * np.sin(thetav) * np.sin(rv)), axis=-1)

    def computeZ(micPositions):
        mp = micPositions
        return np.array([np.exp(np.dot(p, mp[i])*1j) for i in range(len(mp))])

    def computeBeamPattern(z):
        return np.sum(z, axis=0) / z.shape[0]

    def cropac(z1, z2):
        if len(z1.shape) > 1:
            z1 = computeBeamPattern(z1)
        if len(z2.shape) > 1:
            z2 = computeBeamPattern(z2)
        return np.maximum(0, np.real(z1 * np.conj(z2)))
        # return np.maximum(0, np.real(2 * z1 * np.conj(z2)) / np.real(z1 * np.conj(z1) + z2 * np.conj(z2) + 1e-18))

    def plotMesh(ax, z):
        polar_pts, cart_pts, xy = plot_points
        interp = RegularGridInterpolator(xy, np.abs(z), bounds_error=False)
        z = interp(cart_pts)
        c = ax.pcolormesh(*polar_pts, np.abs(z), shading='auto', cmap='viridis')
        plt.colorbar(c, ax=ax, label='Gain (linear)')
    
    idx1 = []
    idx2 = []

    for i in range(len(pattern)):
        if w_checkbox_list1[i].value:
            idx1.append(i)
    for i in range(len(pattern)):
        if w_checkbox_list2[i].value:
            idx2.append(i)

    z = computeZ(pattern)
    z0 = computeBeamPattern(z)

    z_cropac_product = np.ones(z0.shape)
    z_cropac = [np.ones(z0.shape) for _ in range(MAX_CROPAC)]
    for i in range(MAX_CROPAC):
        if str(i) in state.array.cropac:
            ix1, ix2 = state.array.cropac[str(i)]
            if len(ix1) < 1 or len(ix2) < 1:
                continue
            z_cropac[i] = cropac(z[ix1], z[ix2])
            z_cropac_product *= z_cropac[i]

    kr = 0
    if w_cropac_low.value:
        kr, z1, z2, g = low_cropac_beampattern(np.array([1, 0]), z.T, pattern.T, k)
        z_cropac[0] = z1
        z_cropac[1] = z2
        z_cropac_product = g
    else:
        z_cropac[0] = np.sum(z[idx1] / len(idx1), axis=0)
        z_cropac[1] = np.sum(z[idx2] / len(idx2), axis=0)

    

    with out:
        clear_output(wait=True)

        print(kr, jv(1, kr), jv(2, kr))
        fig = plt.figure(figsize=(fig_width, fig_width * 0.55))
        ax1 = fig.add_subplot(2, 3, 1)
        ax2 = fig.add_subplot(2, 3, 2, projection='polar')
        ax3 = fig.add_subplot(2, 3, 3, projection='polar')
        ax4 = fig.add_subplot(2, 3, 4, projection='polar')
        ax5 = fig.add_subplot(2, 3, 5, projection='polar')
        ax6 = fig.add_subplot(2, 3, 6, projection='polar')
        
        rect1 = Rectangle((0,0), 0.277, 0.32, color='gray', fc = 'none',lw = 2)
        rect2 = Rectangle((0.03,0.03), 0.21, 0.26, color='gray', fc = 'none',lw = 1)
        ax1.add_patch(rect1)
        ax1.add_patch(rect2)
        ax1.scatter(patternX[idx], patternY[idx], s=100, c='C1')
        ax1.scatter(patternX, patternY, s=40, c='C0')
        ax1.scatter(patternX[idx1], patternY[idx1], s=5, c='white')
        ax1.scatter(patternX[idx2], patternY[idx2], s=5, c='black')
        for i in range(len(patternX)):
            ax1.annotate(
                str(i),
                (patternX[i], patternY[i]),
                textcoords="offset points",
                xytext=(4, 4),
                fontsize=8,
                color='gray',
                ha='left',
                va='bottom'
            )
        ax1.axis('equal')

        plotMesh(ax2, z0)
        plotMesh(ax3, z0 * z_cropac_product)
        plotMesh(ax6, z_cropac_product)
        plotMesh(ax4, z_cropac[0])
        plotMesh(ax5, z_cropac[1])

        ax2.set_title('DAS beam pattern')
        ax3.set_title('CroPaC beam pattern')
        ax6.set_title('CroPaC gain pattern')
        ax4.set_title('CroPaC pattern U')
        ax5.set_title('CroPaC pattern V')
        plt.tight_layout()
        plt.show()

def add_mic(change=None):
    if not state.do_update:
        return
    state.array.positions.append([w_x.value, w_y.value])
    update_plot()

def delete_mic(change=None):
    if not state.do_update:
        return
    if len(state.array.positions) > 1:
        idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))
        state.array.positions.pop(idx)
        update_plot()

def set_mic_position(change=None):
    if not state.do_update:
        return
    idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))
    state.array.positions[idx] = [w_x.value, w_y.value]
    update_plot()

def set_idx(change=None):
    if not state.do_update:
        return
    idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))
    x, y = state.array.positions[idx]
    state.do_update = False
    w_x.value = x
    w_y.value = y
    state.do_update = True
    update_plot()

def set_cropac_idx(change=None):
    if not state.do_update:
        return
    state.do_update = False
    for i in range(MAX_MICS):
        w_checkbox_list1[i].value = False
        w_checkbox_list2[i].value = False
    if str(w_cropac_idx.value) in state.array.cropac:
        ix1, ix2 = state.array.cropac[str(w_cropac_idx.value)]
        for i in ix1:
            w_checkbox_list1[i].value = True
        for i in ix2:
            w_checkbox_list2[i].value = True
    state.do_update = True
    update_plot()

def set_cropac(change=None):
    if not state.do_update:
        return
    idx1 = []
    idx2 = []
    for i in range(MAX_MICS):
        if w_checkbox_list1[i].value:
            idx1.append(i)
    for i in range(MAX_MICS):
        if w_checkbox_list2[i].value:
            idx2.append(i)

    if len(idx1) < 1 or len(idx2) < 1:
        state.array.cropac.pop(str(w_cropac_idx.value), None)
    else:
        state.array.cropac[str(w_cropac_idx.value)] = [idx1, idx2]
    update_plot()

def set_array(array):
    if not state.do_update:
        return
    state.array.__dict__ = array
    state.do_update = False
    
    w_cropac_idx.value = 0
    w_idx.value = 0
    
    for i in range(MAX_MICS):
        w_checkbox_list1[i].value = False
        w_checkbox_list2[i].value = False
    if "0" in state.array.cropac:
        ix1, ix2 = state.array.cropac["0"]
        for i in ix1:
            w_checkbox_list1[i].value = True
        for i in ix2:
            w_checkbox_list2[i].value = True
    x, y = state.array.positions[0]
    w_x.value = x
    w_y.value = y
    state.do_update = True
    update_plot()

def get_array():
     return state.array.__dict__

w_new.on_click(add_mic)
w_delete.on_click(delete_mic)
w_x.observe(set_mic_position, names="value")
w_y.observe(set_mic_position, names="value")
w_idx.observe(set_idx, names="value")
w_cropac_idx.observe(set_cropac_idx, names="value")

for w in (*w_checkbox_list1, *w_checkbox_list2):
    w.observe(set_cropac, names="value")

for w in (w_f, w_cropac_low):
    w.observe(update_plot, names="value")

afw, loadar = make_array_file_widgets(get_array, set_array)

display(VBox([
    afw,
    w_f,
    HBox([w_idx, w_new, w_delete]),
    w_x,
    w_y,
    HBox([w_cropac_low, w_cropac_idx]),
    HBox(w_checkbox_list1, layout=Layout(margin='0 0 0 100px')),
    HBox(w_checkbox_list2, layout=Layout(margin='0 0 0 100px')),
    out
]))


loadar(None)

In [4]:
a = np.exp(2j * np.pi / 8 * (np.arange(8) + 0.5)) * 0.1
a = np.array([np.real(a) + 0.276 / 2, np.imag(a) + 0.32 / 2])
print(np.sort(np.round(a[0], 3)))
print(np.sort(np.round(a[1], 3)))

[0.046 0.046 0.1   0.1   0.176 0.176 0.23  0.23 ]
[0.068 0.068 0.122 0.122 0.198 0.198 0.252 0.252]
